In [1]:
from google.colab import files
uploaded = files.upload()

Saving 1.jpeg to 1 (2).jpeg
Saving 2.jpeg to 2 (2).jpeg
Saving 3.jpeg to 3 (2).jpeg
Saving 4.jpeg to 4 (2).jpeg
Saving 5.jpeg to 5 (2).jpeg
Saving 6.jpeg to 6 (2).jpeg


In [2]:
!pip install -q diffusers transformers accelerate torch torchvision xformers imageio imageio-ffmpeg

In [3]:
!pip uninstall -y diffusers
!pip install diffusers>=0.27.0

Found existing installation: diffusers 0.36.0
Uninstalling diffusers-0.36.0:
  Successfully uninstalled diffusers-0.36.0


In [4]:
import diffusers
print(diffusers.__version__)

0.36.0


In [5]:
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid",
    torch_dtype=torch.float16,
    variant="fp16"
)

pipe.enable_model_cpu_offload()   # IMPORTANT for T4

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

In [6]:
import os

generator = torch.manual_seed(42)
all_video_paths = []

for idx, img_name in enumerate(uploaded.keys()):
    print(f"Processing {img_name}...")

    image = load_image(img_name)
    image = image.resize((512, 512))  # keep 512 max for T4

    frames = pipe(
        image,
        decode_chunk_size=4,   # lower than 8 (safer)
        generator=generator,
        num_frames=8           # reduced from 12 (safer)
    ).frames[0]

    video_path = f"clip_{idx}.mp4"
    export_to_video(frames, video_path, fps=6)
    all_video_paths.append(video_path)

    torch.cuda.empty_cache()

print("All clips generated!")

Processing 1 (2).jpeg...


  0%|          | 0/25 [00:00<?, ?it/s]

Processing 2 (2).jpeg...


  0%|          | 0/25 [00:00<?, ?it/s]

Processing 3 (2).jpeg...


...

Processing 5 (2).jpeg...


  0%|          | 0/25 [00:00<?, ?it/s]

Processing 6 (2).jpeg...


  0%|          | 0/25 [00:00<?, ?it/s]

All clips generated!


In [7]:
import imageio

final_frames = []

for video in all_video_paths:
    reader = imageio.get_reader(video)
    for frame in reader:
        final_frames.append(frame)

imageio.mimsave("final_video.mp4", final_frames, fps=6)

from google.colab import files
files.download("final_video.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>